# Schubert Winterreise Dataset v2.1 to TiLiA

The full run: download the archive, convert the 24 songs in each of the 9 performances (216 files), run TiLiA, and build the package. Set `DATASET_CONVERTERS_OUT` to the output folder and `TILIA` to the `tilia` executable, and have the repository root on `PYTHONPATH`.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

from common.download import _md5
from swd import convert
from swd.download import fetch_resumable

out = Path(os.environ["DATASET_CONVERTERS_OUT"])
out

## Download

The whole archive (0.7 GB) goes to the download cache. The download continues after a stall, and the archive's md5 is checked.

In [ ]:
archive = fetch_resumable(convert.ARCHIVE_URL, convert.ARCHIVE_MD5)
assert _md5(archive) == convert.ARCHIVE_MD5
print(archive, archive.stat().st_size, "bytes, md5 ok")
source = convert.Source(archive)

## Convert

For each item: its CSVs in `csv/<ID>/`, its TiLiA script in `scripts/<ID>.txt`, run through TiLiA into `tla/<ID>.tla`. HU33 and SC06 load their recordings from the archive (extracted into the cache); the other seven performances have no audio.

In [ ]:
ids = convert.item_ids()
summary = convert.convert_all(source, out, ids, audio=True)
len(summary)

## Package

The package holds copies of the files without media or file paths, with the licence and `NOTICE.md`.

In [ ]:
package = convert.package(out, out / "package")
print(package, package.stat().st_size, "bytes")

## Summary

Components counted in the saved files, against the rows of the source files.

In [ ]:
totals = {layer: [0, 0] for layer in convert.LAYERS}
for item, layers in summary.items():
    for layer, n in layers.items():
        totals[layer][0] += n["components"]
        totals[layer][1] += n["source_rows"]
for layer, (components, rows) in totals.items():
    print(f"{layer:20} components {components:7}  source rows {rows:7}")
different = [(i, l) for i, ls in summary.items() for l, n in ls.items() if n["components"] != n["source_rows"]]
print(len(summary), "files;", len(different), "layers where components differ from source rows")
assert not different